#### Importing CSV table and pandas  
*Note: I used Claude by Anthropic to "fix basic syntax and logic errors" after writing code first.*

In [1]:
import pandas as pd

events = pd.read_csv("events.csv")
events.head(12)

,event_id,event_timestamp,customer_id,product_id,event_type,quantity,unit_price,country,source,status
0,E000157,2026-08-13 13:06:46,C00059,P0027,purchase,2,110.37,MX,mobile,processed
1,E000021,2026-08-13 21:51:05,C00125,P0053,view,1,25.99,US,partner_api,ok
2,E000356,2026-08-08 22:22:55,C00113,P0056,purchase,1,205.10,US,web,ok
3,E000006,2026-08-01 23:43:46,C00032,P0060,view,1,117.56,GB,web,failed
4,E000124,2026-08-12 02:29:55,C00018,P0016,purchase,5,13.80,MX,web,failed
5,E000100,2026-08-01 01:27:26,C00121,P0019,view,1,115.37,US,web,processed
6,E000208,2026-08-02 12:53:22,C00027,P0050,view,1,64.08,DE,partner_api,processed
7,E000070,2026-08-09 20:59:54,C00108,P0035,view,-1,223.45,DE,partner_api,ok
8,E000111,2026-08-11 12:18:26,C00093,P0040,purchase,1,50.96,GB,partner_api,processed
9,E000120,2026-08-17 22:42:30,NaN,P0025,view,1,177.78,CA,mobile,ok


#### Task 1:  Build reusable cleaning functions 
- 1) **standardize** event_type and status by trimming whitespace and converting text to a consistent case;
- 2) **convert** event_timestamp, quantity, and unit_price to appropriate data types;
- 3) **remove** exact duplicates  
- 4) **identify** **invalid** or missing values
- For invalid numeric or timestamp values, convert them to missing values rather than allowing the entire pipeline to fail.

*Note: I broke up each function into smaller parts in order to use them for the pipeline quality summary. The functions are created to be implemented in alphabetic order*

In [2]:
# Part A ----------------------------------------
def standardize(df): 
    out = df.copy()         # copy to modify
    out["event_type"] = (   # event type
        out["event_type"]   
        .astype("string")   # ensures string type
        .str.strip()        # removes whitespace
        .str.lower()        # makes all chr lowercase
    )
    out["status"] = (       # repeats process on status column
            out["status"]
            .astype("string")
            .str.strip()
            .str.lower()
    )
    return out

standardized_test = standardize(events)
print("Ran successfully!")

Ran successfully!


In [3]:
# Part B ----------------------------------------
def convert(df):
    out = df.copy()                                         # copy to modify

    out["event_timestamp"] = pd.to_datetime(                # timestamp = date/time
        out["event_timestamp"], errors = "coerce")    
    out["quantity"] = pd.to_numeric(                        # quantity = int
        out["quantity"], errors= "coerce")
    out["unit_price"] = pd.to_numeric(                      # price = float
        out["unit_price"], errors = "coerce")
    return out

convert_test = convert(events)
print("Ran successfully!")

Ran successfully!


In [4]:
# Part C ----------------------------------------
def remove_dupe(df):
    return df.drop_duplicates()     # output = new df with no duplicates


# difference between length of original data frame and cleaned data frame
removed_count_test = len(events) - len(remove_dupe(events)) 
print(f"Number of removed lines: {removed_count_test}")

Number of removed lines: 12


In [5]:
# Part D ----------------------------------------
def identify_invalid(df):
    out = df.copy()                 # copy to modify

    # returns true if any value in row is invalid
    invalid_timestamp  = out["event_timestamp"].isna()
    invalid_ids        = out["customer_id"].isna() | out["product_id"].isna()
    invalid_quantity   = out["quantity"].isna()    | (out["quantity"] < 0)
    invalid_price      = out["unit_price"].isna()  | (out["unit_price"] < 0)

    # used in pipeline summary
    out["invalid_timestamp"] = invalid_timestamp
    out["invalid_id"]        = invalid_ids

    # used AI to combine all bools into one column instead of multiple columns
    out["is_invalid"] = invalid_timestamp | invalid_ids | invalid_quantity | invalid_price

    return out 

new_events = standardize(events)
new_events = convert(new_events)
new_events = remove_dupe(new_events)
flagged = identify_invalid(new_events)

print(f"Number of invalid lines: {flagged["is_invalid"].sum()}")

Number of invalid lines: 47


#### Task 2: Debugging

In [6]:
%xmode Context
def calculate_revenue (df):
    out = df.copy()
    out = convert(out)
    out["revenue"] = out["quantity"] * out["unit_price"]
    purchases = out[out["event_type"] == "purchase"]
    return purchases.groupby("country")["revenue"].sum()

calculate_revenue(events)

Exception reporting mode: Context


country
CA    6747.91
DE    9616.46
GB    8449.99
MX    6371.25
US    3657.65
Name: revenue, dtype: float64

" %xmode Context " helped me identify the logic error presented in Problem 3     
Problem 1: (Syntax) "Purchaces" ->"purchaces"  
Problem 2: (Syntax) "reveneu" -> "revenue"  
Problem 3: (Logic)
- In the *original* line 3, the function is attempting to multiply strings. 
- I was able to fix this by making a copy of the data frame and running the convert() function defined above in part B.  

#### Task 3
- 1) Python loop solution
- 2) Pandas solution 

In [7]:
def country_revenue_py(df): 
    tot_rev = {}
    for _, row in df.iterrows():                        # collects information by looping              
        if (row["event_type"] == "purchase") and (      # a purchace was made
            row["quantity"] > 0) and (                  # valid purchace quantity
            row["unit_price"] > 0):                     # valid purchace price
            country = row["country"]
            rev = row["quantity"] * row["unit_price"]           # sums the revenue for eahc counttry
            tot_rev[country] = tot_rev.get(country,0) + rev     # adds country and rev to dictionary
    return tot_rev

def country_revenue_pan(df):
    orders = df.copy()            # copy to modify
    valid = orders[                                 # collects information through filtering
        (orders["event_type"] == "purchase") &      # a purchace was made
        (orders["quantity"] > 0) &                  # valid purchace quantity
        (orders["unit_price"] > 0)]                 # valid purchace price

    valid["revenue"] = valid["quantity"] * valid["unit_price"]  # creates new column 
    return valid.groupby("country")["revenue"].sum()            # groups revenue sum by country

Testing, time, and profiling:

In [8]:
test1 = standardize(events)
test1 = convert(test1)
test1 = remove_dupe(test1)
%timeit country_revenue_py(test1)
print(country_revenue_py(test1))

27.6 ms ± 4.45 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
{'MX': 6955.249999999999, 'US': 3704.7800000000007, 'GB': 8692.539999999999, 'CA': 6654.889999999999, 'DE': 9732.48}


In [9]:
test1 = standardize(events)
test1 = convert(test1)
test1 = remove_dupe(test1)
%timeit country_revenue_pan(test1)
print(country_revenue_pan(test1))

4.03 ms ± 285 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
country
CA    6654.89
DE    9732.48
GB    8692.54
MX    6955.25
US    3704.78
Name: revenue, dtype: float64


In [10]:
%prun country_revenue_py(test1)

         59477 function calls (57793 primitive calls) in 0.131 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.016    0.016    0.115    0.115 <string>:1(<module>)
      420    0.016    0.000    0.030    0.000 construction.py:531(sanitize_array)
      420    0.009    0.000    0.076    0.000 series.py:369(__init__)
    16718    0.008    0.000    0.011    0.000 {built-in method builtins.isinstance}
      420    0.005    0.000    0.007    0.000 generic.py:6119(__finalize__)
      943    0.004    0.000    0.020    0.000 series.py:945(__getitem__)
      421    0.004    0.000    0.090    0.000 frame.py:1538(iterrows)
      420    0.004    0.000    0.012    0.000 managers.py:2053(from_array)
      943    0.003    0.000    0.004    0.000 base.py:3601(get_loc)
      943    0.003    0.000    0.010    0.000 series.py:1029(_get_value)
5067/3383    0.003    0.000    0.004    0.000 {built-in method builtins.len}
        1  

In [11]:
%prun country_revenue_pan(test1)

         3934 function calls (3844 primitive calls) in 0.011 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
  821/801    0.000    0.000    0.001    0.000 {built-in method builtins.isinstance}
       24    0.000    0.000    0.000    0.000 generic.py:6119(__finalize__)
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        1    0.000    0.000    0.000    0.000 string_.py:1176(_cmp_method)
        1    0.000    0.000    0.011    0.011 <string>:1(<module>)
       10    0.000    0.000    0.000    0.000 take.py:118(_take_nd_ndarray)
       32    0.000    0.000    0.000    0.000 generic.py:6196(__setattr__)
        8    0.000    0.000    0.000    0.000 managers.py:1226(iget)
        4    0.000    0.000    0.000    0.000 managers.py:2487(_merge_blocks)
        6    0.000    0.000    0.001    0.000 series.py:369(__init__)
       11    0.000    0.000    0.000    0.000 blocks.py:639(cop

1) The method using pandas was faster than the version usng python looping (4.11 ms vs 27.6ms, respectively). 
2) The profiler showed that the loop-based funtion had almost 15x the amount of calls versus the pandas version.  
3) Profiling in data engineering is important because it allows the engineer to understand where the "heavy lifting" of a function is occurring, and understand where scaling affects a process. 

#### Task 4: Pipeline summary  
- 1) number of raw rows
- 2) number of cleaned rows
- 3) number of duplicate rows removed
- 4) number of records with unusable timestampes
- 5) number of records with invalid id's
- 6) total valid revenue

In [14]:
def pipeline_summary (raw_df , clean_df):
    # step 1 - 3 -----------------------
    raw_rows =      len(raw_df)
    clean_rows =    len(clean_df)
    dupes =         len(raw_df) - len(clean_df)

    # step 4 + 5 -----------------------
    invalid = identify_invalid(clean_df)
    invalid_time = invalid["invalid_timestamp"].sum()
    invalid_id   = invalid["invalid_id"].sum()

    # step 6 ---------------------------
    rev = calculate_revenue(clean_df).sum()

    print(f"Number of raw rows: {raw_rows}")
    print(f"Number of cleaned rows: {clean_rows}")
    print(f"Number of duplicate rows removed: {dupes}")
    print(f"Number of unusable timestamps: {invalid_time}")
    print(f"Number of invalid ID's: {invalid_id}")
    print(f"Total valid revenue: {rev}")

# Cleaning the raw
clean = standardize(events)
clean = convert(clean)
clean = remove_dupe(clean)
clean = identify_invalid(clean)

pipeline_summary(events, clean)

Number of raw rows: 432
Number of cleaned rows: 420
Number of duplicate rows removed: 12
Number of unusable timestamps: 10
Number of invalid ID's: 23
Total valid revenue: 35582.76


While many parts of this pipeline meet the criteria for a usable data pipeline, I feel as though more information (like patterns in the sources, timestamps, quantity of purchaces per country, etc.) can be derived form this data frame. Similarly, I would have used a different approach to how the functions are "grouped". This being said, this is a sufficient summary. 